# Trial And Error Colab T4 Playbook
Notebook ini untuk eksperimen banyak konfigurasi IndoBERT+BiLSTM di Colab T4, dengan evaluasi inline otomatis (metrics table + classification report + confusion matrix).

## 0) Setup Runtime
Pilih GPU T4 di Colab: Runtime > Change runtime type > T4 GPU.

In [ ]:
!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt
!nvidia-smi


## 1) Dataset Otomatis dari Repo
Notebook ini pakai dataset dari repo, jadi tidak perlu upload manual.
File default: `data/dataset_relabel_mbg_improved.csv`.

In [ ]:
import os
assert os.path.exists('data/dataset_relabel_mbg_improved.csv'), 'Dataset improved tidak ditemukan di repo'
print('Dataset siap:', 'data/dataset_relabel_mbg_improved.csv')


## 2) Persiapan Data (CPU)
- Split utama 70/30 dari dataset improved
- Preprocess train/test
- Split internal train_sub/val_sub
- Balancing hanya train_sub (untuk training).

In [ ]:
!python3 src/05_split_data.py --input data/dataset_relabel_mbg_improved.csv --text-col text --label-col Labeling_Sentimen --train-ratio 0.7 --val-ratio 0 --test-ratio 0.3
!python3 src/03_preprocess_text.py --train-input data/train.csv --test-input data/test.csv --text-col text


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('data/train.csv')
train_sub, val_sub = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Labeling_Sentimen'].astype(str))
train_sub.to_csv('data/train_sub.csv', index=False, encoding='utf-8-sig')
val_sub.to_csv('data/val_sub.csv', index=False, encoding='utf-8-sig')
print('train_sub', len(train_sub), train_sub['Labeling_Sentimen'].value_counts().to_dict())
print('val_sub  ', len(val_sub), val_sub['Labeling_Sentimen'].value_counts().to_dict())


In [ ]:
!python3 src/05b_balance_train.py --input data/train_sub.csv --label-col Labeling_Sentimen --target-mode fixed --target-count 1500 --output data/train_sub_balanced.csv --log-output outputs/train_sub_balance_log.json


## 3) Daftar Konfigurasi Trial
Default pakai `src/resources/step7_push80_valid_trials.json` (agresif untuk dorong macro-F1).
Jika OOM, fallback ke `src/resources/step7_top3_minority_boost.json`.


In [ ]:
import json, pandas as pd
TRIAL_PACK_PATH = 'src/resources/step7_push80_valid_trials.json'
pack = json.load(open(TRIAL_PACK_PATH, encoding='utf-8'))
display(pd.DataFrame(pack)[['trial_name','model_name','max_len','batch_size','hidden_size','freeze_bert','loss_type','lr','epochs']])


## 4) Jalankan Trial Satu-per-Satu + Evaluasi Inline
Cell ini akan:
1) train 1 trial
2) evaluate di test asli
3) simpan artefak per trial
4) tampilkan ringkasan metrik inline

In [ ]:
import os, json, shutil, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
# os.environ['HF_TOKEN'] = 'hf_xxx'
pack = json.load(open('src/resources/step7_push80_valid_trials.json', encoding='utf-8'))
results = []
Path('outputs/figures').mkdir(parents=True, exist_ok=True)
# Optional: batasi trial yang dijalankan jika mau cepat
TRIAL_NAMES_TO_RUN = [t['trial_name'] for t in pack]  # bisa diubah ke subset
for trial in pack:
    if trial['trial_name'] not in TRIAL_NAMES_TO_RUN:
        continue
    trial_name = trial['trial_name']
    tmp_json = Path('outputs') / f'tmp_{trial_name}.json'
    tmp_json.write_text(json.dumps([trial], ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'\n=== RUN: {trial_name} ===')
    cmd_train = [
        'python3', 'src/07_indobert_bilstm.py',
        '--train', 'data/train_sub_balanced.csv',
        '--val', 'data/val_sub.csv',
        '--trial-configs-json', str(tmp_json),
        '--max-trials', '1'
    ]
    train_proc = subprocess.run(cmd_train, text=True, capture_output=True)
    if train_proc.returncode != 0:
        print(train_proc.stdout)
        print(train_proc.stderr)
        results.append({'trial_name': trial_name, 'status': 'train_failed'})
        continue
    cmd_eval = [
        'python3', 'src/09_evaluate.py',
        '--test', 'data/test.csv',
        '--model-path', 'models/best_indobert_bilstm.pt'
    ]
    eval_proc = subprocess.run(cmd_eval, text=True, capture_output=True)
    if eval_proc.returncode != 0:
        print(eval_proc.stdout)
        print(eval_proc.stderr)
        results.append({'trial_name': trial_name, 'status': 'eval_failed'})
        continue
    metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
    report = pd.read_csv('outputs/classification_report.csv')
    # archive artifacts per trial
    Path('outputs').mkdir(exist_ok=True)
    Path('outputs').joinpath(f'final_metrics_{trial_name}.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
    shutil.copy('outputs/classification_report.csv', f'outputs/classification_report_{trial_name}.csv')
    shutil.copy('outputs/confusion_matrix.csv', f'outputs/confusion_matrix_{trial_name}.csv')
    if Path('outputs/figures/confusion_matrix.png').exists():
        shutil.copy('outputs/figures/confusion_matrix.png', f'outputs/figures/confusion_matrix_{trial_name}.png')
    row = {
        'trial_name': trial_name,
        'model_name': trial['model_name'],
        'max_len': trial['max_len'],
        'batch_size': trial['batch_size'],
        'hidden_size': trial['hidden_size'],
        'freeze_bert': trial['freeze_bert'],
        'loss_type': trial['loss_type'],
        'accuracy': metrics.get('accuracy'),
        'precision_macro': metrics.get('precision_macro'),
        'recall_macro': metrics.get('recall_macro'),
        'f1_macro': metrics.get('f1_macro'),
        'status': 'ok'
    }
    results.append(row)
    print('Metrics:', {k: row[k] for k in ['accuracy','precision_macro','recall_macro','f1_macro']})
    display(report)
    cm_img = Path(f'outputs/figures/confusion_matrix_{trial_name}.png')
    if cm_img.exists():
        display(Image(filename=str(cm_img)))
results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values('f1_macro', ascending=False, na_position='last')
    results_df.to_csv('outputs/trial_and_error_colab_results.csv', index=False, encoding='utf-8-sig')
    display(results_df)
    print('Saved: outputs/trial_and_error_colab_results.csv')


## 5) Pilih Model Terbaik
Model terbaik adalah baris dengan `f1_macro` tertinggi di tabel hasil trial.
Artefak tersimpan di `outputs/final_metrics_<trial>.json` dan `outputs/classification_report_<trial>.csv`.